In [1]:
import numpy as np
import pandas as pd

In [ ]:
def calc_burned_by_class(df, mtbs_class):
    """
    Sum burned area in hectares for a single MTBS severity class.

    Args:
        df (pd.DataFrame): Burned area records with columns mtbs_class,
            state_name, access, and area_km2.
        mtbs_class (int): MTBS severity class to filter on.

    Returns:
        pd.DataFrame: Columns state_name, access, and class_{mtbs_class}_ha.
    """
    # Keep only rows for this severity class, then total area per state/access pair
    grp = df[df['mtbs_class'] == mtbs_class].groupby(
        ['state_name', 'access'], as_index=False
    )['area_km2'].sum()

    # 1 km2 = 100 ha
    grp['area_ha'] = grp['area_km2'] * 100

    # Tag the area column with its class so repeated merges don't collide
    return grp[['state_name', 'access', 'area_ha']].rename(
        columns={'area_ha': f'class_{mtbs_class}_ha'}
    )


### File I/O

In [ ]:
# Load in the dataset
data_folder = "<PATH/TO/PROJECT/FOLDER>"


In [ ]:
# Read the CSV files
severity_df = pd.read_csv(f'{data_folder}/data/tables/mtbs_nfs_access_class_summary.csv')

# Read in the extent dataframe
extent_df = pd.read_csv(f'{data_folder}/data/tables/nfs_access_extent_km2.csv')
extent_df["extent_ha"] = extent_df["extent_masked_km2"] * 100
extent_df = extent_df[["state_name", "access_class", "extent_ha"]]


In [ ]:
# Filter for the recent decade and severity classes 1-4
period_df = severity_df[
    severity_df['year'].between(2014, 2023) &
    severity_df['mtbs_class'].isin([1, 2, 3, 4])
].copy()


### Calculate burned area by state, access class, and severity class

In [ ]:
# Start from the extent table; standardize the join key name and copy so the
# original extent_df is left untouched by the merges below
summary = extent_df.rename(columns={'access_class': 'access'}).copy()

for cls in [1, 2, 3, 4]:
    # Left join keeps every state/access combination in the extent table,
    # including those with no burned area in this class
    summary = summary.merge(
        calc_burned_by_class(period_df, cls),
        on=['state_name', 'access'], how='left'
    )

# State/access pairs missing from a class group burned 0 ha, not unknown
summary = summary.fillna(0)


### Calculate percentages

In [ ]:
for cls in [1, 2, 3, 4]:
    summary[f'pct_{cls}'] = summary[f'class_{cls}_ha'] / summary['extent_ha'] * 100

summary['pct_all']           = summary[['pct_1', 'pct_2', 'pct_3', 'pct_4']].sum(axis=1)
summary['pct_low_mod_high']  = summary[['pct_2', 'pct_3', 'pct_4']].sum(axis=1)
summary['pct_mod_high']      = summary[['pct_3', 'pct_4']].sum(axis=1)



### Calculate ALL row (totals across all states)

In [ ]:
all_rows = []
for access in ['roaded', 'roadless', 'wilderness']:
    ext = extent_df[extent_df['access_class'] == access]['extent_ha'].sum()
    sub = summary[summary['access'] == access]
    row = {'state_name': 'ALL', 'access': access, 'extent_ha': ext}
    for cls in [1, 2, 3, 4]:
        row[f'class_{cls}_ha'] = sub[f'class_{cls}_ha'].sum()
        row[f'pct_{cls}'] = row[f'class_{cls}_ha'] / ext * 100 if ext > 0 else 0
    row['pct_all']          = sum(row[f'pct_{cls}'] for cls in [1, 2, 3, 4])
    row['pct_low_mod_high'] = sum(row[f'pct_{cls}'] for cls in [2, 3, 4])
    row['pct_mod_high']     = sum(row[f'pct_{cls}'] for cls in [3, 4])
    all_rows.append(row)

result = pd.concat([summary, pd.DataFrame(all_rows)], ignore_index=True)


### Format and display

In [ ]:
access_order  = ['roaded', 'roadless', 'wilderness']
access_labels = {'roaded': 'Developed', 'roadless': 'IRA', 'wilderness': 'Wilderness'}

state_order = sorted(result[result['state_name'] != 'ALL']['state_name'].unique())
state_rank  = {s: i for i, s in enumerate(state_order + ['ALL'])}
result['_state_rank']  = result['state_name'].map(state_rank)
result['_access_rank'] = result['access'].map({a: i for i, a in enumerate(access_order)})
result = result.sort_values(['_state_rank', '_access_rank']).reset_index(drop=True)

result['all_ha']          = result[['class_1_ha', 'class_2_ha', 'class_3_ha', 'class_4_ha']].sum(axis=1)
result['low_mod_high_ha'] = result[['class_2_ha', 'class_3_ha', 'class_4_ha']].sum(axis=1)
result['mod_high_ha']     = result[['class_3_ha', 'class_4_ha']].sum(axis=1)

def pct_fmt(pct):
    return f"{pct:.1f}%"

result_display = pd.DataFrame({
    'State':                        result['state_name'],
    'Access Class':                 result['access'].map(access_labels),
    'Forest Area (ha)':             result['extent_ha'].round(0).astype(int).apply(lambda x: f"{x:,}"),
    'Unburned to Low':              result['pct_1'].apply(pct_fmt),
    'Low':                          result['pct_2'].apply(pct_fmt),
    'Moderate':                     result['pct_3'].apply(pct_fmt),
    'High':                         result['pct_4'].apply(pct_fmt),
    'All Severity':                 result['pct_all'].apply(pct_fmt),
    'Low, Moderate, or High':       result['pct_low_mod_high'].apply(pct_fmt),
    'Moderate or High':             result['pct_mod_high'].apply(pct_fmt),
})

result_display


,State,Access Class,Forest Area (ha),Unburned to Low,Low,Moderate,High,All Severity,"Low, Moderate, or High",Moderate or High
0,Arizona,Developed,"1,943,897",5.3%,12.8%,3.1%,1.0%,22.2%,16.9%,4.1%
1,Arizona,IRA,"220,617",5.0%,12.4%,5.9%,2.2%,25.5%,20.6%,8.1%
2,Arizona,Wilderness,"241,322",4.8%,11.6%,8.6%,2.5%,27.5%,22.7%,11.1%
3,California,Developed,"4,301,027",3.3%,11.3%,10.2%,10.7%,35.5%,32.2%,20.8%
4,California,IRA,"887,139",4.7%,13.7%,13.0%,10.0%,41.4%,36.7%,23.0%
5,California,Wilderness,"1,120,003",7.6%,16.8%,12.3%,8.1%,44.8%,37.2%,20.4%
6,Colorado,Developed,"2,314,039",0.9%,1.6%,1.3%,1.3%,5.0%,4.2%,2.5%
7,Colorado,IRA,"1,285,555",1.0%,2.0%,2.4%,1.9%,7.3%,6.3%,4.3%
8,Colorado,Wilderness,"777,926",0.7%,1.4%,1.6%,1.9%,5.6%,4.9%,3.5%
9,Idaho,Developed,"2,520,875",1.2%,2.4%,1.7%,1.4%,6.7%,5.5%,3.1%


In [ ]:
result_display.to_csv(f"{data_folder}/data/tables/nfs_mtbs_severity_summary_2014_2023.csv", index=False)
